In [2]:
pip install pyarrow

     --------------------------------------- 26.2/26.2 MB 31.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Parquet File Reader

Quick notebook to read and explore parquet files in the data directory.

## Usage
1. Set the `file_path` variable to your parquet file
2. Optionally set `num_rows` and `columns_to_show`
3. Run all cells

In [3]:
import pandas as pd
from pathlib import Path

# Set display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

## Configuration

Update these variables with your file path and preferences:

**Note:** Since this notebook is in the `data/` directory, file paths should be relative to `data/`.
- ✅ Correct: `'options/etfs/SPY_20251011_205633.parquet'`
- ❌ Wrong: `'data/options/etfs/SPY_20251011_205633.parquet'`

In [4]:
# File to read (relative to data/ directory)
# Since this notebook is IN the data/ directory, just use the path from here
file_path = 'options/etfs/IWM_20251011_205633.parquet'

# Number of rows to display
num_rows = 100

# Specific columns to show (leave as None to show all columns)
# Example: columns_to_show = ['symbol', 'strike', 'lastPrice', 'delta', 'gamma']
columns_to_show = None

## Load File

In [5]:
# Resolve file path
# Since this notebook is in the data/ directory, paths are relative to data/
file_path = Path(file_path)

# If the file doesn't exist as-is, it might already have 'data/' prefix - try without it
if not file_path.exists() and str(file_path).startswith('data'):
    # Remove the 'data/' prefix since we're already in the data directory
    parts = file_path.parts
    if parts[0] == 'data':
        file_path = Path(*parts[1:])

if not file_path.exists():
    # Try to find where we are and provide helpful error
    import os
    print(f"Current directory: {os.getcwd()}")
    print(f"Looking for: {file_path}")
    print(f"Absolute path: {file_path.absolute()}")
    raise FileNotFoundError(f"File not found: {file_path}\\n\\nMake sure the path is relative to the data/ directory.\\nExample: 'options/etfs/SPY_20251011_205633.parquet'")

print(f"Reading: {file_path}")

# Read parquet file
df = pd.read_parquet(file_path)

print(f"✓ Loaded {len(df):,} rows")

Reading: options\etfs\IWM_20251011_205633.parquet
✓ Loaded 2,770 rows


## File Info

In [6]:
print(f"📊 File Info:")
print(f"  File: {file_path.name}")
print(f"  Total rows: {len(df):,}")
print(f"  Total columns: {len(df.columns)}")
print(f"  File size: {file_path.stat().st_size / 1024:.1f} KB")
print(f"  Memory usage: {df.memory_usage(deep=True).sum() / 1024 / 1024:.1f} MB")

📊 File Info:
  File: IWM_20251011_205633.parquet
  Total rows: 2,770
  Total columns: 27
  File size: 270.7 KB
  Memory usage: 1.6 MB


## Column Information

In [7]:
print(f"📋 Columns ({len(df.columns)}):")
print()

col_info = []
for col in df.columns:
    dtype = df[col].dtype
    non_null = df[col].notna().sum()
    null_pct = (df[col].isna().sum() / len(df) * 100)
    col_info.append({
        'Column': col,
        'Type': str(dtype),
        'Non-Null': f"{non_null:,}",
        'Null %': f"{null_pct:.1f}%"
    })

col_df = pd.DataFrame(col_info)
display(col_df)

📋 Columns (27):



,Column,Type,Non-Null,Null %
0,symbol,object,"2,770",0.0%
1,expiration,datetime64[ns],"2,770",0.0%
2,optionType,object,"2,770",0.0%
3,contractSymbol,object,"2,770",0.0%
4,strike,float64,"2,770",0.0%
5,currency,object,"2,770",0.0%
6,lastPrice,float64,"2,770",0.0%
7,change,float64,"2,770",0.0%
8,percentChange,float64,"2,770",0.0%
9,volume,float64,"2,770",0.0%


## Data Preview

In [8]:
# Filter columns if specified
df_display = df.copy()

if columns_to_show:
    available_cols = [col for col in columns_to_show if col in df.columns]
    missing_cols = [col for col in columns_to_show if col not in df.columns]
    
    if missing_cols:
        print(f"⚠️  Missing columns: {', '.join(missing_cols)}")
    
    if available_cols:
        df_display = df_display[available_cols]
        print(f"✓ Showing only columns: {', '.join(available_cols)}")
    else:
        raise ValueError("None of the specified columns exist in the file")

print(f"\nFirst {min(num_rows, len(df_display))} rows:")
display(df_display.head(num_rows))


First 100 rows:


,symbol,expiration,optionType,contractSymbol,strike,currency,lastPrice,change,percentChange,volume,openInterest,bid,ask,contractSize,lastTradeDate,impliedVolatility,inTheMoney,snapshot_datetime,snapshot_date,snapshot_time,market_session,underlying_price,delta,gamma,theta,vega,rho
0,IWM,2025-10-13,calls,IWM251013C00212000,212.0,USD,33.85,0.000000,0.000000,1.0,0,26.35,26.55,REGULAR,2025-10-07 14:54:38,1.101567,True,2025-10-11 20:56:33.388729-04:00,2025-10-11,20:56:33.388729,CLOSE,237.79,0.971872,0.004432,-0.001217,0.000085,6.330677e-05
1,IWM,2025-10-13,calls,IWM251013C00220000,220.0,USD,18.08,-3.940001,-17.892828,11.0,1,18.37,18.56,REGULAR,2025-10-10 20:09:32,0.825685,True,2025-10-11 20:56:33.388729-04:00,2025-10-11,20:56:33.388729,CLOSE,237.79,0.957420,0.008311,-0.001280,0.000120,6.470500e-05
2,IWM,2025-10-13,calls,IWM251013C00226000,226.0,USD,20.05,0.000000,0.000000,1.0,1,12.46,12.60,REGULAR,2025-10-03 19:55:40,0.625980,True,2025-10-11 20:56:33.388729-04:00,2025-10-11,20:56:33.388729,CLOSE,237.79,0.931110,0.016038,-0.001411,0.000175,6.460645e-05
3,IWM,2025-10-13,calls,IWM251013C00228000,228.0,USD,12.97,-6.880000,-34.659950,1.0,1,10.46,10.63,REGULAR,2025-10-10 17:39:59,0.553227,True,2025-10-11 20:56:33.388729-04:00,2025-10-11,20:56:33.388729,CLOSE,237.79,0.917478,0.020824,-0.001430,0.000201,6.421584e-05
4,IWM,2025-10-13,calls,IWM251013C00230000,230.0,USD,8.65,-6.350000,-42.333336,26.0,5,8.54,8.68,REGULAR,2025-10-10 20:14:23,0.504399,True,2025-10-11 20:56:33.388729-04:00,2025-10-11,20:56:33.388729,CLOSE,237.79,0.886511,0.028857,-0.001633,0.000254,6.253145e-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,IWM,2025-10-14,calls,IWM251014C00249000,249.0,USD,0.03,-0.380000,-92.682930,459.0,907,0.02,0.03,REGULAR,2025-10-10 18:09:35,0.236336,False,2025-10-11 20:56:33.388729-04:00,2025-10-11,20:56:33.388729,CLOSE,237.79,0.005722,0.003801,-0.000046,0.000030,7.878935e-07
96,IWM,2025-10-14,calls,IWM251014C00250000,250.0,USD,0.02,-0.240000,-92.307690,1288.0,1398,0.00,0.02,REGULAR,2025-10-10 19:57:18,0.242195,False,2025-10-11 20:56:33.388729-04:00,2025-10-11,20:56:33.388729,CLOSE,237.79,0.003637,0.002475,-0.000031,0.000020,5.008021e-07
97,IWM,2025-10-14,calls,IWM251014C00251000,251.0,USD,0.01,-0.150000,-93.749990,685.0,122,0.00,0.02,REGULAR,2025-10-10 19:24:22,0.257820,False,2025-10-11 20:56:33.388729-04:00,2025-10-11,20:56:33.388729,CLOSE,237.79,0.003233,0.002092,-0.000030,0.000018,4.451227e-07
98,IWM,2025-10-14,calls,IWM251014C00252000,252.0,USD,0.02,-0.070000,-77.777790,223.0,1376,0.01,0.02,REGULAR,2025-10-10 19:23:44,0.273445,False,2025-10-11 20:56:33.388729-04:00,2025-10-11,20:56:33.388729,CLOSE,237.79,0.002917,0.001798,-0.000029,0.000016,4.015306e-07


## Summary Statistics

In [9]:
# Show statistics for numeric columns
numeric_cols = df_display.select_dtypes(include=['number']).columns

if len(numeric_cols) > 0:
    print(f"Summary Statistics ({len(numeric_cols)} numeric columns):")
    display(df_display[numeric_cols].describe())
else:
    print("No numeric columns to display statistics for.")

Summary Statistics (15 numeric columns):


,strike,lastPrice,change,percentChange,volume,openInterest,bid,ask,impliedVolatility,underlying_price,delta,gamma,theta,vega,rho
count,2770.000000,2770.000000,2770.000000,2770.000000,2770.000000,2770.000000,2770.000000,2770.000000,2770.000000,2.770000e+03,2770.000000,2770.000000,2770.000000,2770.000000,2.770000e+03
mean,220.482671,21.576357,-0.237715,26.553659,619.477978,3963.827798,20.049325,21.165935,0.333263,2.377900e+02,0.141561,0.009092,-0.000177,0.003185,8.912122e-04
std,53.874986,30.319489,2.344015,107.062533,4822.488073,12083.450002,29.287040,30.165552,0.283518,2.842684e-14,0.552285,0.014093,0.000310,0.003298,7.832305e-03
min,85.000000,0.010000,-35.099995,-98.611110,0.000000,0.000000,0.000000,0.000000,0.000010,2.377900e+02,-1.000000,0.000000,-0.005473,0.000000,-5.766383e-02
25%,190.000000,1.410000,-0.067500,-6.064457,2.000000,21.000000,1.140000,1.402500,0.229843,2.377900e+02,-0.158843,0.001651,-0.000203,0.000532,-8.240703e-04
50%,224.000000,7.350000,0.000000,0.000000,6.000000,189.000000,6.325000,7.405000,0.284065,2.377900e+02,0.001408,0.004559,-0.000096,0.002126,2.244265e-07
75%,250.000000,30.655000,0.050000,8.928293,60.000000,1594.750000,27.245000,29.432500,0.379080,2.377900e+02,0.693631,0.010056,-0.000035,0.004768,3.033829e-03
max,365.000000,179.970000,7.760000,1082.352900,129658.000000,171901.000000,172.570000,173.400000,5.833987,2.377900e+02,1.000000,0.162614,0.000134,0.014306,2.932666e-02


## Quick Analysis

Run custom analysis on the loaded dataframe:

In [10]:
# Example: Show unique values for categorical columns
categorical_cols = df.select_dtypes(include=['object', 'category']).columns

if len(categorical_cols) > 0:
    print("Unique values in categorical columns:")
    for col in categorical_cols[:5]:  # Show first 5 categorical columns
        unique_count = df[col].nunique()
        print(f"\n{col}: {unique_count} unique values")
        if unique_count <= 10:
            print(f"  Values: {df[col].unique().tolist()}")

Unique values in categorical columns:

symbol: 1 unique values
  Values: ['IWM']

optionType: 2 unique values
  Values: ['calls', 'puts']

contractSymbol: 2770 unique values

currency: 1 unique values
  Values: ['USD']

contractSize: 1 unique values
  Values: ['REGULAR']


## Custom Analysis

Add your own analysis here. The dataframe is available as `df`:

In [11]:
# Your custom analysis here
# Example for options data:

if 'optionType' in df.columns:
    print("Options Breakdown:")
    print(df['optionType'].value_counts())

if 'symbol' in df.columns:
    print("\nSymbols:")
    print(df['symbol'].value_counts())

if 'expiration' in df.columns:
    print("\nExpirations:")
    print(df['expiration'].value_counts().head(10))

Options Breakdown:
optionType
calls    1412
puts     1358
Name: count, dtype: int64

Symbols:
symbol
IWM    2770
Name: count, dtype: int64

Expirations:
expiration
2025-10-17    202
2026-01-16    197
2025-11-21    167
2025-12-19    149
2025-12-31    142
2025-10-24    135
2026-03-31    131
2026-06-18    129
2025-10-31    129
2027-01-15    124
Name: count, dtype: int64


## Export Subset (Optional)

Uncomment to export filtered data to CSV:

In [12]:
# Uncomment to export
# output_file = 'exported_data.csv'
# df_display.head(num_rows).to_csv(output_file, index=False)
# print(f"✓ Exported to {output_file}")